In [ ]:
## Load Data --Docs--Divide our documents into chunks documents -- text -- vectors-- vector Embedding

In [ ]:
##  SImple GenAI app using langchian and openai
import os
from dotenv import load_dotenv
load_dotenv

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')   #langsmith tracking
os.environ["LANGCHAIN_TRACING_V2"] = "true"  #langsmith tracing
os.environ["LANGCHAIN_PROJECT"] = os.getenv('LANGCHAIN_PROJECT')  #langsmith project name

In [ ]:
## Data Ingestion -- From the website we need to scrap the data
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://docs.smith.langchian.com/tutorials")

In [ ]:
docs=loader.load()
docs

In [ ]:
## Load Data --Docs-- Dividde our text into chunks --text --vectors --vector Embedding --vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)
documents

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()


In [ ]:
from langchain_community.vectorstores import FAISS
vectorstredb=FAISS.from_documents(documents, embeddings)
vectorstredb

In [ ]:
## Query from vector storedb
query="Langsmith has 2 usage limits: total traces and extended"
result=vectorstredb.similarity_search(query)
result[0].page_content

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [ ]:
## Retrieval Chain, Document Chain


from langchain.chains.combine_documents import create_stuff_document_chain
from langchain.core.prompts import ChatPromptTemplate

prompts = ChatPromptTemplate.from_template(
    """Answer the question based on the following context: {context} and if you don't know the answer say you don't know. Question: {question}
    """
)

document_chain=create_stuff_document_chain(llm,prompts)
document_chain


In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"langsmith has 2 usage limits: total traces and extended"
    "context":[Document(page_comtent="LangSmith has 2 usage limits: total traces and extended")]
})

In [ ]:
## INput ---Retriever --vectorstoredb

vectorstredb
retriever=vectorstredb.as_retriever()

from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [ ]:
## Get the response from the llm
response=retrieval_chain.invoke({
    "question":"Langsmith has 2 usage limits: total traces and extended"
})
response=response['output']
response